<p align="center"><b>Assignment 9</b></p>

Write a program to implement the local binary pattern on gray scale
image.



In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("input.jpg", 0)

if img is None:
    print("Image not found!")
    exit()

padded = np.pad(img, 1, mode='constant')
result = np.zeros(img.shape, dtype=np.uint8)

for i in range(img.shape[0]):
    for j in range(img.shape[1]):

        center = padded[i+1, j+1]

        neighbors = [
            padded[i, j],
            padded[i, j+1],
            padded[i, j+2],
            padded[i+1, j+2],
            padded[i+2, j+2],
            padded[i+2, j+1],
            padded[i+2, j],
            padded[i+1, j]
        ]

        value = 0

        for k in range(8):
            if neighbors[k] >= center:
                value += 2**(7-k)

        result[i,j] = value

plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(result, cmap='gray')
plt.title("LBP Image")
plt.axis("off")

plt.show()


Write a program to denoise the input image corrupted by Gaussian and
Speckle noise by using following filters (3x3 or 5x5 window).

(a) Weiner filter

(b) Median filtering

(c) Gaussian filter

(d) Bilateral filter

In [ ]:
#Read image and geenerating noise
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("input.jpg", 0)

if img is None:
    print("Image not found!")
    exit()

n = int(input("Enter window size (3 or 5): "))

if n != 3 and n != 5:
    print("Invalid window size!")
    exit()

p = n // 2

original = img.astype(float)

gaussian_noise = np.random.normal(0, 15, img.shape)
gaussian_noisy = np.clip(original + gaussian_noise, 0, 255)

speckle_noise = np.random.normal(0, 0.15, img.shape)
speckle_noisy = np.clip(original + original * speckle_noise, 0, 255)


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt


img = cv2.imread('image.jpg', 0)

if img is None:
    print("Image not found")
    exit()

img = img.astype(float)


gaussian_noise = np.random.normal(0, 20, img.shape)
gaussian_img = np.clip(img + gaussian_noise, 0, 255)

speckle_noise = np.random.normal(0, 0.2, img.shape)
speckle_img = np.clip(img + img * speckle_noise, 0, 255)



def wiener_filter(image, size=5):
    pad = size // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)

    noise_var = np.var(image - np.mean(image))

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = padded[i:i+size, j:j+size]

            mean = np.mean(window)
            var = np.var(window)

            result[i, j] = mean + max(0, var-noise_var) / (var+1e-8) * (image[i, j]-mean)

    return np.clip(result, 0, 255)



def median_filter(image, size=5):
    pad = size // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = padded[i:i+size, j:j+size]
            result[i, j] = np.median(window)

    return result



def gaussian_filter(image, size=5, sigma=1):
    pad = size // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)


    kernel = np.zeros((size, size))

    for i in range(size):
        for j in range(size):
            x = i - pad
            y = j - pad
            kernel[i, j] = np.exp(-(x*x+y*y)/(2*sigma*sigma))

    kernel = kernel / np.sum(kernel)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = padded[i:i+size, j:j+size]
            result[i, j] = np.sum(window * kernel)

    return np.clip(result, 0, 255)



def bilateral_filter(image, size=5, sigma_s=2, sigma_r=30):
    pad = size // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):

            window = padded[i:i+size, j:j+size]

            spatial = np.zeros((size, size))
            for x in range(size):
                for y in range(size):
                    dx = x - pad
                    dy = y - pad
                    spatial[x, y] = np.exp(
                        -(dx*dx + dy*dy)/(2*sigma_s*sigma_s)
                    )

            intensity = np.exp(
                -((window - image[i, j])**2)/(2*sigma_r*sigma_r)
            )

            weights = spatial * intensity

            result[i, j] = np.sum(weights * window) / np.sum(weights)

    return np.clip(result, 0, 255)


g1 = wiener_filter(gaussian_img)
g2 = median_filter(gaussian_img)
g3 = gaussian_filter(gaussian_img)
g4 = bilateral_filter(gaussian_img)

s1 = wiener_filter(speckle_img)
s2 = median_filter(speckle_img)
s3 = gaussian_filter(speckle_img)
s4 = bilateral_filter(speckle_img)

def display(images, titles):
    plt.figure(figsize=(12, 8))

    for i in range(len(images)):
        plt.subplot(2, 3, i+1)
        plt.imshow(images[i], cmap='gray')
        plt.title(titles[i])
        plt.axis('off')

    plt.tight_layout()
    plt.show()


titles = ['Original', 'Noisy', 'Wiener',
          'Median', 'Gaussian', 'Bilateral']

display([img, gaussian_img, g1, g2, g3, g4], titles)

display([img, speckle_img, s1, s2, s3, s4], titles)
